# GPT-5.4 Prompting Guide

## 1. Introduction

GPT-5.4 is a frontier model in the GPT-5 family designed for complex professional workloads such as coding, document analysis, structured data extraction, research synthesis, and multi-step agent workflows. It builds on prompting patterns established in GPT-5 and GPT-5.2 while benefiting from newer platform capabilities that support long-running workflows and tool-integrated execution.

In practice, most prompt structures that work well for GPT-5.2 continue to work well for GPT-5.4. However, developers often observe the greatest gains when prompts explicitly define task boundaries, output contracts, and verification expectations.

This guide focuses on practical prompt patterns that help developers obtain reliable behavior from GPT-5.4 in production environments.

> **Scope note**
> This guide synthesizes GPT-5-family prompting practices together with GPT-5.4 platform capabilities. It is intended as a practical companion to the official API documentation. Recommendations should be validated with evaluation workflows when used in production systems.


# 2. Key behavioral differences

Compared with earlier GPT-5 models, GPT-5.4 is commonly used in workflows that involve:

* longer contexts
* stronger integration with tools
* multi-step task execution
* structured outputs for downstream systems

These characteristics affect how prompts should be written.

### Long-context workflows

GPT-5-class models support large context windows, enabling tasks such as:

* multi-document reasoning
* long conversation histories
* extended agent workflows

Prompts should anchor claims to specific source sections and avoid generic summarization instructions when precise grounding is required.

### Tool-integrated execution

Many GPT-5 deployments rely on external tools such as search systems, databases, or code execution environments. Prompt instructions should clearly define when the model should rely on tools rather than internal reasoning.

### Structured multi-step tasks

When GPT-5.4 is used inside agent systems or automation pipelines, prompts benefit from clearly defined task phases, output contracts, and verification rules.


# 3. Prompting patterns

The following patterns improve reliability across many GPT-5-class deployments.

These are structured prompt design techniques rather than model-specific tricks.


## 3.1 Controlling verbosity and output shape

Explicit length constraints improve response consistency and reduce unnecessary verbosity.

Example:

```
<output_verbosity_spec>
- Default: 3–6 sentences or ≤5 bullets.
- For simple questions: ≤2 sentences.
- For complex tasks:
  - 1 overview paragraph
  - ≤5 bullets summarizing results or changes.
- Avoid repeating the user’s request.
- Prefer concise explanations over long narrative paragraphs.
</output_verbosity_spec>
```

This pattern is useful when responses must fit strict formats such as APIs or structured reports.


## 3.2 Preventing scope drift

Models capable of complex reasoning may expand the task unless prompts define boundaries explicitly.

Example:

```
<scope_constraints>
- Implement exactly what the user requested.
- Do not introduce additional features or analysis unless explicitly requested.
- Preserve existing interfaces and structure when editing code or documents.
- If ambiguity exists, choose the simplest interpretation consistent with the request.
</scope_constraints>
```

Scope constraints are particularly valuable in coding and document editing tasks.


## 3.3 Instruction ordering

Complex prompts should present instructions in priority order.

Example:

```
<role_and_priority>
You are an implementation assistant for technical workflows.

Follow instructions in this order:
1. Satisfy the user’s request.
2. Follow required output format and schema.
3. Use tools or provided context when needed.
4. Avoid unnecessary commentary.
5. Ask clarifying questions only if essential information is missing.
</role_and_priority>
```

Ordered instructions help reduce conflicts between formatting rules and task instructions.


## 3.4 Handling ambiguity and hallucination risk

When prompts lack critical information, models may infer details incorrectly.

Mitigation pattern:

```
<ambiguity_handling>
- If the request is ambiguous, identify the missing information.
- Ask concise clarifying questions when necessary.
- Do not invent specific numbers, references, or citations.
- If assumptions are required, state them explicitly.
</ambiguity_handling>
```

This pattern is particularly important for compliance-sensitive tasks.


# 4. Compaction (Extending Effective Context)

Long-running conversations may eventually exceed practical context limits.

GPT-5-class systems support **conversation compaction**, which compresses earlier conversation state into a smaller representation while preserving task-relevant information.

Typical use cases include:

* long research workflows
* agent systems with many tool calls
* extended analysis sessions

Compaction is usually applied after major milestones rather than after every message.

Example:


In [ ]:
from openai import OpenAI
import json

client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",
    input=[{
        "role": "user",
        "content": "Draft a migration plan for a legacy billing system."
    }]
)

output_json = [item.model_dump() for item in response.output]

compacted_response = client.responses.compact(
    model="gpt-5.4",
    input=[
        {
            "role": "user",
            "content": "Draft a migration plan for a legacy billing system."
        },
        output_json[0]
    ]
)

print(json.dumps(compacted_response.model_dump(), indent=2))


Compacted outputs should generally be treated as opaque system state rather than human-readable summaries.


# 5. Agentic steerability & user updates

Multi-step workflows benefit from structured progress updates.

Example:

```
<user_updates_spec>
Provide brief updates when:
- starting a new phase of work
- discovering information that changes the plan
- user confirmation is required

Each update should include one concrete outcome.
</user_updates_spec>
```

Planning rules can also improve task stability:

```
<plan_revision_rules>
- Begin with a short plan for multi-step tasks.
- Revise the plan only when new evidence changes priorities.
- Preserve completed work whenever possible.
</plan_revision_rules>
```

These patterns maintain transparency without unnecessary verbosity.


# 6. Tool-calling and parallelism

Tool use improves reliability when tasks require fresh data or external state.

Best practices include:

* describing each tool clearly
* preferring tools when data may be outdated
* verifying state-changing operations
* parallelizing independent retrieval tasks

Example:

```
<tool_usage_rules>
- Prefer tools when fresh or external data is required.
- Parallelize independent read operations.
- Confirm parameters before write operations.
- After writes, verify the resulting state.
</tool_usage_rules>
```

Explicit tool rules reduce incorrect assumptions.


# 7. Structured extraction, PDF, and Office workflows

Structured extraction tasks benefit from explicit schemas.

Example:

```
<extraction_spec>
Extract structured data into JSON.

Schema:
{
  "document_id": string,
  "counterparty": string | null,
  "effective_date": string | null,
  "financial_amount": string | null
}

Rules:
- Follow the schema exactly.
- Use null for missing values.
- Do not guess missing data.
</extraction_spec>
```

Providing a schema significantly improves extraction accuracy.


# 8. Prompt migration to GPT-5.4

When migrating prompts from earlier models, follow an evaluation-driven process.

Typical workflow:

1. Switch models without modifying the prompt.
2. Pin the `reasoning_effort` parameter explicitly.
3. Run evaluation tests to establish a baseline.
4. Adjust prompt structure only if regressions appear.
5. Re-evaluate after each change.

This process isolates the effect of each modification.


# 9. Web search and research

Research workflows benefit from explicit evidence requirements.

Example:

```
<web_research_rules>
- Base claims on reliable sources.
- Prefer primary or official references.
- Cite sources for factual statements.
- If sources disagree, explain the discrepancy.
</web_research_rules>
```

Structured research prompts improve answer traceability.


# 10. Conclusion

GPT-5.4 continues the GPT-5 family’s emphasis on structured reasoning, tool integration, and multi-step task execution.

Effective prompting typically involves:

* explicit scope definition
* structured output contracts
* disciplined tool usage
* evaluation-driven iteration

Teams that combine clear prompts with systematic testing generally achieve the most reliable results.


# Limitations

This guide does not claim undocumented model internals or training details. Some recommendations are derived from general GPT-5 prompting practices and may vary depending on the application.

Production systems should validate prompt behavior using representative evaluation tasks before adopting any pattern at scale.


# Appendix

### Example research agent prompt

```
You are a rigorous research assistant.

Rules:
- Base claims on verifiable evidence.
- Cite reliable sources for factual statements.
- Distinguish facts from inference.
- Organize responses with clear Markdown sections.

Before answering, confirm that:
- claims are supported by evidence
- contradictions are addressed
- the response matches the requested depth.
```
